# 02 — Resultados Preliminares (Modelo Baseline)
**Objetivo:** Estabelecer uma linha de base com uma implementação simples do modelo content-based.
Os resultados desta etapa servem de referência para os ajustes feitos no notebook 03.


In [ ]:
import json, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
DATASET = "Sports_and_Outdoors.jsonl"
SAMPLE  = 150_000
TOP_K   = 10


## 1. Carregamento e Pré-processamento Simples

In [ ]:
records = []
with open(DATASET, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= SAMPLE: break
        try: records.append(json.loads(line))
        except: pass

df = pd.DataFrame(records)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
df['rating']    = pd.to_numeric(df['rating'], errors='coerce')
df['text']      = df['text'].fillna('')
df['title']     = df['title'].fillna('')

# Filtros básicos
prod_counts = df['asin'].value_counts()
df = df[df['asin'].isin(prod_counts[prod_counts >= 5].index)]
df = df.drop_duplicates(subset=['user_id','asin'], keep='last')

print(f"Registros: {len(df):,} | Produtos: {df['asin'].nunique():,} | Usuários: {df['user_id'].nunique():,}")


## 2. Modelo Baseline — TF-IDF Simples

In [ ]:
# Perfil de produto: concatena os textos brutos por produto
product_profiles = (df.groupby('asin')
                      .agg(profile=('text', lambda x: ' '.join(x)),
                           avg_rating=('rating','mean'),
                           n_reviews=('asin','count'))
                      .reset_index())

# TF-IDF básico (sem bigrama, sem stemming)
tfidf_baseline = TfidfVectorizer(max_features=8000, stop_words='english')
matrix_baseline = tfidf_baseline.fit_transform(product_profiles['profile'])

asin_to_idx = {a: i for i, a in enumerate(product_profiles['asin'])}
idx_to_asin = {i: a for a, i in asin_to_idx.items()}

print(f"Matriz TF-IDF baseline: {matrix_baseline.shape}")


## 3. Função de Recomendação Baseline

In [ ]:
def recommend_baseline(user_id, df_train, top_k=TOP_K):
    user_data = df_train[df_train['user_id'] == user_id]
    positive  = user_data[user_data['rating'] >= 4]
    if len(positive) == 0:
        positive = user_data
    consumed = set(user_data['asin'])
    valid = [a for a in positive['asin'] if a in asin_to_idx]
    if not valid:
        return pd.DataFrame()

    indices = [asin_to_idx[a] for a in valid]
    # FIX: np.asarray() converte np.matrix -> ndarray (exigido pelo cosine_similarity)
    profile = np.asarray(matrix_baseline[indices].mean(axis=0))
    sims    = cosine_similarity(profile, matrix_baseline).flatten()
    for a in consumed:
        if a in asin_to_idx:
            sims[asin_to_idx[a]] = -1

    top_idx   = sims.argsort()[::-1][:top_k]
    top_asins = [idx_to_asin[i] for i in top_idx]
    return product_profiles[product_profiles['asin'].isin(top_asins)].copy()

# Demonstração
sample_user = df['user_id'].value_counts().index[0]
recs = recommend_baseline(sample_user, df)
print(f"Recomendacoes para usuario {sample_user}:")
print(recs[['asin','avg_rating','n_reviews']].to_string(index=False))


## 4. Avaliação do Baseline

In [ ]:
def temporal_split(df):
    df = df.sort_values(['user_id','timestamp'])
    train, test = [], []
    for uid, grp in df.groupby('user_id'):
        n_test = max(1, int(len(grp) * 0.2))
        if len(grp) <= n_test:
            train.append(grp)
        else:
            train.append(grp.iloc[:-n_test])
            test.append(grp.iloc[-n_test:])
    return pd.concat(train), pd.concat(test) if test else pd.DataFrame()

df_train, df_test = temporal_split(df)

def precision_recall_at_k(df_train, df_test, rec_fn, k=TOP_K, max_users=300):
    relevant_map = (df_test[df_test['rating'] >= 4]
                    .groupby('user_id')['asin'].apply(set).to_dict())
    users = [u for u in list(relevant_map)[:max_users] if u in df_train['user_id'].values]
    P, R, H = [], [], []
    for u in users:
        recs = rec_fn(u, df_train, top_k=k)
        if recs.empty: continue
        rec_set = set(recs['asin'])
        rel_set = relevant_map[u]
        hits = rec_set & rel_set
        P.append(len(hits) / k)
        R.append(len(hits) / len(rel_set) if rel_set else 0)
        H.append(1 if hits else 0)
    return {'n_users': len(P),
            f'Precision@{k}': np.mean(P),
            f'Recall@{k}':    np.mean(R),
            f'HitRate@{k}':   np.mean(H)}

print(f"Avaliando baseline (K={TOP_K}, ate 300 usuarios)...")
baseline_metrics = precision_recall_at_k(df_train, df_test, recommend_baseline)
print()
print("=== RESULTADOS BASELINE ===")
for k, v in baseline_metrics.items():
    val = f"{v:,}" if isinstance(v, int) else f"{v:.4f}"
    print(f"  {k:<20}: {val}")


## 5. Análise dos Resultados Preliminares

**Observações sobre o baseline:**
- Pré-processamento mínimo (apenas stopwords padrão do scikit-learn)
- Perfil de usuário = média simples dos vetores (sem ponderar pelo rating)
- TF-IDF sem bigramas e com poucos features (8.000)
- Esses resultados serão usados como referência para o ajuste no próximo notebook


In [ ]:
import json
with open("baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f)
print("baseline_metrics.json salvo.")
